# Cleaning the news data with standard NLP preprocessing techniques

We're gonna use the Natural Language Toolkit (nltk) library for this

In [15]:
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [16]:
import pandas as pd
import numpy as np
import re


In [17]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")  # for lemmatization

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vince\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vince\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vince\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vince\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

We are going to simplify our text data with the following methods:
1. Remove stopwords
    - We will remove common words that do not contribute much meaning to the text
2. Tokenization
    - We will split the text into individual words or tokens
3. Lemmatization
    - We will reduce words to their base or root form, so this converts things like "running" to "run"

We chose to do lemmatization over stemming because lemmatization considers the context and converts the word to its meaningful base form, which is more suitable for our sentiment analysis task.

We need to customize our list of stopwords to not remove words that are important for sentiment analysis, such as "not", "no", "never", etc.

These are apparently in a lot of stopword lists by default.

In [18]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

finance_whitelist = {
    "up",
    "down",
    "above",
    "below",
    "under",
    "over",
    "rise",
    "fall",
    "not",
    "no",
    "nor",
    "neither",
    "never",
    "none",
    "more",
    "most",
    "few",
    "less",
}

stop_words.difference_update(finance_whitelist)

The NLTK includes Punkt which is supposed to be better than using split for tokenization so we will use that as well

In [19]:
def clean_text(text):
    # 1. Lowercase all text
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # 3. Tokenize using Punkt
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # 5. Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [ ]:
df_news = pd.read_csv("../../data/news/filtered/all_filtered.csv")

df_news["clean_summary"] = df_news["summary"].apply(clean_text)

df_news.head()

,date,summary,source,adj_date,cleaned_summary
0,2020-07-17 19:51:00,Jim Cramer: A better way to invest in the Covi...,cnbc,2020-07-20,jim cramer better way invest covid19 vaccine g...
1,2020-07-17 19:33:00,Cramer's lightning round: I would own Teradyne...,cnbc,2020-07-20,cramers lightning round would teradyne mad mon...
2,2020-07-17 19:25:00,"Cramer's week ahead: Big week for earnings, ev...",cnbc,2020-07-20,cramers week ahead big week earnings even bigg...
3,2020-07-17 16:24:00,IQ Capital CEO Keith Bliss says tech and healt...,cnbc,2020-07-20,iq capital ceo keith bliss say tech healthcare...
4,2020-07-16 19:36:00,Wall Street delivered the 'kind of pullback I'...,cnbc,2020-07-17,wall street delivered kind pullback ive waitin...


In [ ]:
df_news_clean = df_news[["adj_date", "clean_summary"]].copy()

df_news_clean.to_csv("../../data/news/cleaned/all_clean.csv", index=False)

Now we have our preprocessed text data ready to be vectorized!